In [1]:
!pip install -q flask redis pika joblib pandas scikit-learn requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.3/165.3 kB 7.9 MB/s eta 0:00:00


In [3]:
from google.colab import files

uploaded = files.upload()

Saving faculty_burnout_model_pipeline.pkl to faculty_burnout_model_pipeline.pkl


In [4]:
import pandas as pd
import numpy as np
import json
import joblib
import redis
import pika
from flask import Flask, request, jsonify

# Load the trained ML pipeline
MODEL_PATH = "/content/faculty_burnout_model_pipeline.pkl"

model = joblib.load(MODEL_PATH)

print("All libraries imported successfully!")
print("ML model loaded successfully!")
print("Model path:", MODEL_PATH)

All libraries imported successfully!
ML model loaded successfully!
Model path: /content/faculty_burnout_model_pipeline.pkl


In [7]:
from getpass import getpass

# =========================
# Redis Configuration
# =========================
REDIS_HOST = "129.153.75.221"
REDIS_PORT = 6379
REDIS_USERNAME = "default"
REDIS_DB = 0

REDIS_PASSWORD = getpass("Enter Redis password: ")

# =========================
# RabbitMQ Configuration
# =========================
RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"
RABBITMQ_VHOST = "/"

RABBITMQ_PASSWORD = getpass("Enter RabbitMQ password: ")

# Queue used in Phase 3
QUEUE_NAME = "faculty_burnout_queue"

print("\nConfiguration loaded successfully!")
print("Redis Host:", REDIS_HOST)
print("Redis Port:", REDIS_PORT)
print("RabbitMQ Host:", RABBITMQ_HOST)
print("RabbitMQ Port:", RABBITMQ_PORT)
print("RabbitMQ Queue:", QUEUE_NAME)

Enter Redis password: ··········
Enter RabbitMQ password: ··········

Configuration loaded successfully!
Redis Host: 129.153.75.221
Redis Port: 6379
RabbitMQ Host: 129.153.75.221
RabbitMQ Port: 5672
RabbitMQ Queue: faculty_burnout_queue


In [8]:
# =========================
# Redis Connection
# =========================

redis_client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    username=REDIS_USERNAME,
    password=REDIS_PASSWORD,
    db=REDIS_DB,
    decode_responses=True
)

# Test Redis
redis_client.ping()
print("Redis Connection: SUCCESS")


# =========================
# RabbitMQ Connection
# =========================

rabbitmq_credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

rabbitmq_params = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=rabbitmq_credentials,
    heartbeat=60,
    blocked_connection_timeout=30
)

rabbitmq_connection = pika.BlockingConnection(rabbitmq_params)
rabbitmq_channel = rabbitmq_connection.channel()

# Make sure the queue exists
rabbitmq_channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print("RabbitMQ Connection: SUCCESS")
print("Queue:", QUEUE_NAME)

Redis Connection: SUCCESS
RabbitMQ Connection: SUCCESS
Queue: faculty_burnout_queue


In [9]:
app = Flask(__name__)

FEATURE_COLUMNS = [
    "Teaching_Hours",
    "Advising_Students",
    "Committee_Count",
    "Research_Hours",
    "Admin_Hours",
    "Semester_Progress",
    "Historical_Leave_Days"
]

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()

        # Create cache key using faculty ID
        faculty_id = data.get("Faculty_ID", "UNKNOWN")
        cache_key = f"faculty_burnout:{faculty_id}"

        # -------------------------
        # 1. Check Redis Cache
        # -------------------------
        cached_result = redis_client.get(cache_key)

        if cached_result:
            result = json.loads(cached_result)

            result["source"] = "Redis Cache"

            return jsonify(result), 200

        # -------------------------
        # 2. Cache MISS → ML Model
        # -------------------------
        input_data = pd.DataFrame(
            [[data[column] for column in FEATURE_COLUMNS]],
            columns=FEATURE_COLUMNS
        )

        prediction = model.predict(input_data)[0]

        probabilities = model.predict_proba(input_data)[0]

        class_names = model.classes_

        risk_probabilities = {
            str(class_name): round(float(probability), 4)
            for class_name, probability
            in zip(class_names, probabilities)
        }

        # -------------------------
        # 3. Prepare Result
        # -------------------------
        result = {
            "Faculty_ID": faculty_id,
            "predicted_burnout_risk": str(prediction),
            "risk_probabilities": risk_probabilities
        }

        # -------------------------
        # 4. Store Result in Redis
        # -------------------------
        redis_client.set(
            cache_key,
            json.dumps(result),
            ex=3600
        )

        result["source"] = "ML Model"

        return jsonify(result), 200

    except Exception as e:
        return jsonify({
            "error": str(e)
        }), 500


print("Flask application created successfully!")
print("Endpoint: POST /predict")

Flask application created successfully!
Endpoint: POST /predict


In [11]:
# Create a fresh Flask application
app = Flask("faculty_burnout_api")

FEATURE_COLUMNS = [
    "Teaching_Hours",
    "Advising_Students",
    "Committee_Count",
    "Research_Hours",
    "Admin_Hours",
    "Semester_Progress",
    "Historical_Leave_Days"
]

@app.route("/predict", methods=["POST"])
def faculty_burnout_predict():
    try:
        data = request.get_json()

        if not data:
            return jsonify({
                "error": "Request body is empty"
            }), 400

        # Faculty ID
        faculty_id = data.get("Faculty_ID", "UNKNOWN")

        # Redis cache key
        cache_key = f"faculty_burnout:{faculty_id}"

        # ==================================
        # 1. CHECK REDIS CACHE
        # ==================================
        cached_result = redis_client.get(cache_key)

        if cached_result:
            result = json.loads(cached_result)

            result["source"] = "Redis Cache"

            print("CACHE HIT")
            print("Cache Key:", cache_key)

            return jsonify(result), 200

        print("CACHE MISS")
        print("Cache Key:", cache_key)

        # ==================================
        # 2. ML MODEL PREDICTION
        # ==================================
        input_data = pd.DataFrame(
            [[data[column] for column in FEATURE_COLUMNS]],
            columns=FEATURE_COLUMNS
        )

        prediction = model.predict(input_data)[0]

        probabilities = model.predict_proba(input_data)[0]

        class_names = model.classes_

        risk_probabilities = {
            str(class_name): round(float(probability), 4)
            for class_name, probability
            in zip(class_names, probabilities)
        }

        # ==================================
        # 3. CREATE RESULT
        # ==================================
        result = {
            "Faculty_ID": faculty_id,
            "predicted_burnout_risk": str(prediction),
            "risk_probabilities": risk_probabilities
        }

        # ==================================
        # 4. STORE RESULT IN REDIS
        # ==================================
        redis_client.set(
            cache_key,
            json.dumps(result),
            ex=3600
        )

        print("Result stored in Redis.")

        # ==================================
        # 5. PUBLISH TO RABBITMQ
        # ==================================
        message_body = json.dumps(result)

        rabbitmq_channel.basic_publish(
            exchange="",
            routing_key=QUEUE_NAME,
            body=message_body,
            properties=pika.BasicProperties(
                delivery_mode=2
            )
        )

        print("Message published to RabbitMQ.")

        # ==================================
        # 6. RETURN RESPONSE
        # ==================================
        result["source"] = "ML Model"

        return jsonify(result), 200

    except Exception as e:
        print("ERROR:", str(e))

        return jsonify({
            "error": str(e)
        }), 500


print("Flask application recreated successfully!")
print("Endpoint: POST /predict")
print("Endpoint function: faculty_burnout_predict")

Flask application recreated successfully!
Endpoint: POST /predict
Endpoint function: faculty_burnout_predict


In [12]:
import threading
import time

def run_flask():
    app.run(
        host="0.0.0.0",
        port=8001,
        debug=False,
        use_reloader=False
    )

flask_thread = threading.Thread(
    target=run_flask,
    daemon=True
)

flask_thread.start()

time.sleep(3)

print("Flask server started successfully!")
print("Running on port: 8001")

 * Serving Flask app 'faculty_burnout_api'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8001
 * Running on http://172.28.0.12:8001
INFO:werkzeug:Press CTRL+C to quit


Flask server started successfully!
Running on port: 8001


In [14]:
# Phase 5 - Fix RabbitMQ publishing inside Flask

import json
import pika
import pandas as pd
from flask import Flask, request, jsonify

# Create a fresh Flask application
app = Flask("faculty_burnout_api_fixed")

FEATURE_COLUMNS = [
    "Teaching_Hours",
    "Advising_Students",
    "Committee_Count",
    "Research_Hours",
    "Admin_Hours",
    "Semester_Progress",
    "Historical_Leave_Days"
]


def publish_to_rabbitmq(message):
    """Create a fresh RabbitMQ connection, publish, and close it."""

    credentials = pika.PlainCredentials(
        RABBITMQ_USERNAME,
        RABBITMQ_PASSWORD
    )

    params = pika.ConnectionParameters(
        host=RABBITMQ_HOST,
        port=RABBITMQ_PORT,
        virtual_host=RABBITMQ_VHOST,
        credentials=credentials,
        heartbeat=60,
        blocked_connection_timeout=30,
        connection_attempts=3,
        retry_delay=2
    )

    connection = pika.BlockingConnection(params)
    channel = connection.channel()

    channel.queue_declare(
        queue=QUEUE_NAME,
        durable=True
    )

    channel.basic_publish(
        exchange="",
        routing_key=QUEUE_NAME,
        body=json.dumps(message),
        properties=pika.BasicProperties(
            delivery_mode=2
        )
    )

    connection.close()

    print("RabbitMQ message published successfully!")


@app.route("/predict", methods=["POST"])
def faculty_burnout_predict():

    try:
        data = request.get_json()

        if not data:
            return jsonify({"error": "Request body is empty"}), 400

        faculty_id = data.get("Faculty_ID")

        if not faculty_id:
            return jsonify({"error": "Faculty_ID is required"}), 400

        cache_key = f"faculty_burnout:{faculty_id}"

        # ------------------------------------------------
        # 1. CHECK REDIS CACHE
        # ------------------------------------------------

        cached_result = redis_client.get(cache_key)

        if cached_result:

            result = json.loads(cached_result)
            result["source"] = "Redis Cache"

            print("CACHE HIT")
            print("Cache Key:", cache_key)

            return jsonify(result), 200

        print("CACHE MISS")
        print("Cache Key:", cache_key)

        # ------------------------------------------------
        # 2. PREPARE ML INPUT
        # ------------------------------------------------

        input_data = pd.DataFrame(
            [[data[column] for column in FEATURE_COLUMNS]],
            columns=FEATURE_COLUMNS
        )

        # ------------------------------------------------
        # 3. ML PREDICTION
        # ------------------------------------------------

        prediction = model.predict(input_data)[0]

        probabilities = model.predict_proba(input_data)[0]

        class_names = model.classes_

        risk_probabilities = {
            str(class_name): round(float(probability), 4)
            for class_name, probability
            in zip(class_names, probabilities)
        }

        result = {
            "Faculty_ID": faculty_id,
            "predicted_burnout_risk": str(prediction),
            "risk_probabilities": risk_probabilities
        }

        # ------------------------------------------------
        # 4. STORE RESULT IN REDIS
        # ------------------------------------------------

        redis_client.set(
            cache_key,
            json.dumps(result),
            ex=3600
        )

        print("Result stored in Redis.")

        # ------------------------------------------------
        # 5. PUBLISH EVENT TO RABBITMQ
        # ------------------------------------------------

        publish_to_rabbitmq(result)

        # ------------------------------------------------
        # 6. RETURN RESPONSE
        # ------------------------------------------------

        result["source"] = "ML Model"

        return jsonify(result), 200

    except Exception as e:

        print("ERROR:", str(e))

        return jsonify({
            "error": str(e)
        }), 500


print("Flask application updated successfully!")
print("RabbitMQ publisher uses a fresh connection for every message.")
print("Endpoint: POST /predict")

Flask application updated successfully!
RabbitMQ publisher uses a fresh connection for every message.
Endpoint: POST /predict


In [15]:
import threading
import time

def run_fixed_flask():
    app.run(
        host="0.0.0.0",
        port=8002,
        debug=False,
        use_reloader=False
    )

flask_thread_fixed = threading.Thread(
    target=run_fixed_flask,
    daemon=True
)

flask_thread_fixed.start()

time.sleep(3)

print("Corrected Flask server started!")
print("Running on port: 8002")

 * Serving Flask app 'faculty_burnout_api_fixed'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8002
 * Running on http://172.28.0.12:8002
INFO:werkzeug:Press CTRL+C to quit


Corrected Flask server started!
Running on port: 8002


In [16]:
import requests

test_payload = {
    "Faculty_ID": "FAC1001",
    "Teaching_Hours": 18,
    "Advising_Students": 8,
    "Committee_Count": 2,
    "Research_Hours": 10,
    "Admin_Hours": 4,
    "Semester_Progress": 0.65,
    "Historical_Leave_Days": 3
}

# Remove previous cached result
redis_client.delete("faculty_burnout:FAC1001")

print("Sending request to Flask...")
print()

response = requests.post(
    "http://127.0.0.1:8002/predict",
    json=test_payload
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

Sending request to Flask...

CACHE MISS
Cache Key: faculty_burnout:FAC1001
Result stored in Redis.


INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:45:37] "POST /predict HTTP/1.1" 200 -


RabbitMQ message published successfully!
Status Code: 200
Response:
{'Faculty_ID': 'FAC1001', 'predicted_burnout_risk': 'Low', 'risk_probabilities': {'High': 0.0, 'Low': 0.7146, 'Medium': 0.2854}, 'source': 'ML Model'}


In [17]:
import requests

print("Sending the same request again...")
print()

response = requests.post(
    "http://127.0.0.1:8002/predict",
    json=test_payload
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

INFO:werkzeug:127.0.0.1 - - [18/Sep/2026 14:46:18] "POST /predict HTTP/1.1" 200 -


Sending the same request again...

CACHE HIT
Cache Key: faculty_burnout:FAC1001
Status Code: 200
Response:
{'Faculty_ID': 'FAC1001', 'predicted_burnout_risk': 'Low', 'risk_probabilities': {'High': 0.0, 'Low': 0.7146, 'Medium': 0.2854}, 'source': 'Redis Cache'}


In [18]:
import json
import pika

# Create a separate RabbitMQ connection for the consumer
consumer_credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

consumer_params = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=consumer_credentials,
    heartbeat=60,
    blocked_connection_timeout=30,
    connection_attempts=3,
    retry_delay=2
)

consumer_connection = pika.BlockingConnection(consumer_params)
consumer_channel = consumer_connection.channel()

consumer_channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

# Get one message from the queue
method, properties, body = consumer_channel.basic_get(
    queue=QUEUE_NAME,
    auto_ack=False
)

if method:
    received_message = json.loads(body.decode("utf-8"))

    print("RabbitMQ message received successfully!")
    print()
    print("Received Message:")
    print(json.dumps(received_message, indent=4))

    # Acknowledge the message
    consumer_channel.basic_ack(
        delivery_tag=method.delivery_tag
    )

    print()
    print("Message ACK completed successfully!")

else:
    print("No message available in the queue.")

consumer_connection.close()

RabbitMQ message received successfully!

Received Message:
{
    "Faculty_ID": "FAC1001",
    "predicted_burnout_risk": "Low",
    "risk_probabilities": {
        "High": 0.0,
        "Low": 0.7146,
        "Medium": 0.2854
    }
}

Message ACK completed successfully!


In [19]:
# Phase 5 - Process RabbitMQ message

if "received_message" in globals():

    processed_result = {
        "Faculty_ID": received_message["Faculty_ID"],
        "Burnout_Risk": received_message["predicted_burnout_risk"],
        "Risk_Probabilities": received_message["risk_probabilities"],
        "Processing_Status": "Processed Successfully"
    }

    print("Message processing completed successfully!")
    print()
    print("Processed Result:")
    print(json.dumps(processed_result, indent=4))

else:
    print("No received message found.")
    print("Please run the RabbitMQ consumer cell first.")

Message processing completed successfully!

Processed Result:
{
    "Faculty_ID": "FAC1001",
    "Burnout_Risk": "Low",
    "Risk_Probabilities": {
        "High": 0.0,
        "Low": 0.7146,
        "Medium": 0.2854
    },
    "Processing_Status": "Processed Successfully"
}


In [20]:
print("=" * 60)
print("PHASE 5 - END-TO-END INTEGRATION VERIFICATION")
print("=" * 60)

# 1. Model
print("1. ML MODEL")
print("   ✓ Random Forest model loaded")

# 2. Redis
redis_status = redis_client.ping()
print("2. REDIS")
print("   ✓ Redis Connection:", "SUCCESS" if redis_status else "FAILED")

# 3. Cached result
cache_key = "faculty_burnout:FAC1001"
cached_data = redis_client.get(cache_key)

print("   ✓ Cache Result:", "AVAILABLE" if cached_data else "NOT FOUND")

# 4. RabbitMQ
rabbit_credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

rabbit_params = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=rabbit_credentials,
    heartbeat=60,
    blocked_connection_timeout=30,
    connection_attempts=3,
    retry_delay=2
)

verification_connection = pika.BlockingConnection(rabbit_params)
verification_channel = verification_connection.channel()

verification_channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print("3. RABBITMQ")
print("   ✓ RabbitMQ Connection: SUCCESS")
print("   ✓ Queue:", QUEUE_NAME)

verification_connection.close()

# 5. Processed result
print("4. MESSAGE PROCESSING")

if "processed_result" in globals():
    print("   ✓ Consumer Processing: SUCCESS")
    print("   ✓ Processing Status:",
          processed_result["Processing_Status"])
else:
    print("   ⚠ Consumer Processing: Not available")

# 6. Final result
print()
print("=" * 60)
print("PHASE 5 STATUS: COMPLETED")
print("=" * 60)

print()
print("Integration Flow:")
print("Flask API")
print("   ↓")
print("Redis Cache Check")
print("   ↓")
print("ML Model (on cache miss)")
print("   ↓")
print("Redis Store")
print("   ↓")
print("RabbitMQ Publish")
print("   ↓")
print("RabbitMQ Consumer")
print("   ↓")
print("Message Processing")
print("   ↓")
print("API Response")

PHASE 5 - END-TO-END INTEGRATION VERIFICATION
1. ML MODEL
   ✓ Random Forest model loaded
2. REDIS
   ✓ Redis Connection: SUCCESS
   ✓ Cache Result: AVAILABLE
3. RABBITMQ
   ✓ RabbitMQ Connection: SUCCESS
   ✓ Queue: faculty_burnout_queue
4. MESSAGE PROCESSING
   ✓ Consumer Processing: SUCCESS
   ✓ Processing Status: Processed Successfully

PHASE 5 STATUS: COMPLETED

Integration Flow:
Flask API
   ↓
Redis Cache Check
   ↓
ML Model (on cache miss)
   ↓
Redis Store
   ↓
RabbitMQ Publish
   ↓
RabbitMQ Consumer
   ↓
Message Processing
   ↓
API Response
